# Verifica con Nagini del numero di porte: algoritmo di Grover

 ## Notazione
 - n: numero di qubit del circuito (qubit di stato)
 - m: numero di qubit su cui agisce la riflessione, cioè `len(qubits)`
 - |O|: numero di porte dell'oracolo
 - k: numero di iterazioni di Grover  

## Premessa


- Cominciamo ad analizzare il codice di `GroverOperator`, che costruisce una **singola iterazione di Grover**
- Una singola iterazione applica in ordine:
    - l'oracolo, che marca le soluzioni con un segno -1
    - la trasformata di Hadamard $H^{⊗n}$
    - uno spostamento di fase condizionale, che lascia invariato $|0…0⟩$ e cambia segno a tutti gli altri stati
    - un'ulteriore trasformata di Hadamard $H^{⊗n}$

- Qiskit scrive l'iterazione come $Q = A S_0 A† S_f$, dove:
    -  $S_f$ è l'oracolo, 
    - $A = H^{⊗n}$ è la preparazione di stato 
    - $S_0$ è lo spostamento di fase condizionale (riflessione attorno a $|0…0⟩$)
- Gli ultimi tre passi insieme, $A S_0 A†$, formano l'inversione attorno alla media $2|ψ⟩⟨ψ| - I$


### Struttura di `GroverOperator`

(file `grover_originale/grover_operator_originale.py`)

- `__init__`: 
    - salva i parametri (oracolo, preparazione di stato, riflessione, qubit di riflessione, `insert_barriers`, `mcx_mode`) e chiama `_build`. 
    - Se l'oracolo o la riflessione sono dati come `Statevector` o `Operator`, li converte in un circuito `Diagonal`

- `reflection_qubits`: 
    - i qubit su cui agisce la riflessione $S_0$ di default tutti i qubit di stato, cioè `[0, ..., n-1]`
- `zero_reflection`: 
    - restituisce $S_0$. 
    - Se l'utente non l'ha fornita, la costruisce chiamando la funzione `_zero_reflection`
- `state_preparation`: 
    - restituisce A. 
    - Se l'utente non l'ha fornita, costruisce un circuito con H sui qubit di riflessione
- `oracle`: 
    - restituisce l'oracolo passato dall'utente, che marca con -1 le soluzioni. 
- `_build`: 
    - compone l'iterazione completa nell'ordine oracolo, $A†$ (`state_preparation.inverse()`), $S_0, A$
    - imposta la fase globale a $π$ per compensare il segno di $S_0$ e infine impacchetta tutto in un unico gate con `to_gate()`

**Ordine della verifica**
- Si parte dalle funzioni che non dipendono da altre: 
    1. `_zero_reflection` 
    2. la preparazione di stato
    3. Poi `_build`, che le usa tramite le loro postcondizioni
   4. Infine `construct_circuit` della classe `Grover`, che ripete k volte l'iterazione

# 1. `_zero_reflection`

- Parto da questa funzione perchè non è richiamata da altre.
- Costruisce $S_0$, la riflessione attorno a $|0…0⟩$ sui qubit indicati. 
    - l'operazione da svolgere è $2|0⟩⟨0| - I$ 
    - viene impelentato $I - 2|0⟩⟨0|$ perchè è più semplice e perchè differisce di una fase globale (stato quantistico è definito a meno di una fase globale) 


## 1.1 Codice originale

File `grover_originale/grover_operator_originale.py`, righe 289-311 (Qiskit, licenza Apache 2.0).

In [ ]:
def _zero_reflection(
    num_state_qubits: int, qubits: List[int], mcx_mode: Optional[str] = None
) -> QuantumCircuit:
    qr_state = QuantumRegister(num_state_qubits, "state")
    reflection = QuantumCircuit(qr_state, name="S_0")

    num_ancillas = MCXGate.get_num_ancilla_qubits(len(qubits) - 1, mcx_mode)
    if num_ancillas > 0:
        qr_ancilla = AncillaRegister(num_ancillas, "ancilla")
        reflection.add_register(qr_ancilla)
    else:
        qr_ancilla = AncillaRegister(0)

    reflection.x(qubits)
    if len(qubits) == 1:
        reflection.z(0)  # MCX does not allow 0 control qubits, therefore this is separate
    else:
        reflection.h(qubits[-1])
        reflection.mcx(qubits[:-1], qubits[-1], qr_ancilla[:], mode=mcx_mode)
        reflection.h(qubits[-1])
    reflection.x(qubits)

    return reflection

## 1.2 Limite da dimostrare  

3 porte se m = 1 e 2m + 3 porte se m ≥ 2

- il numero di porte dipende solo da m, non dal numero totale di qbit del circuito

- In entrambi i casi il circuito è racchiuso tra due strati di X (bit flip) su tutti i qubit di riflessione: 2m porte
    - servono a scambiare |0…0⟩ e |1…1⟩, così il cambio di segno finisce su |0…0⟩
 
- In mezzo serve una porta che cambi segno solo a |1…1⟩ (phase flip)
- **Caso m = 1**: basta una Z sul qubit
    - totale: 2 X + 1 Z = 2m + 1 = 3 porte
- **Caso m ≥ 2**: serve una Z multi-controllata (controlli: tutti i qubit tranne l'ultimo; target: l'ultimo)
    - Qiskit la costruisce come H, MCX, H sul target, grazie all'uguaglianza HXH = Z
    - totale: 2m X + H + MCX + H = 2m + 3 porte

## 1.3 Metodi aggiunti allo stub
- il punto precedente fa vedere che vanno aggiunti dei metodi allo stub che era stato fatto per Deutsch. 
- conteneva x, h, cx 

**METODO Z** identico a h

In [ ]:
def z(self, qubit: int) -> None:
    Requires(Acc(self.num_qubits, 1/2) and Acc(self.gates))
    Requires(0 <= qubit and qubit < self.num_qubits)

    Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))
    Ensures(self.gates == Old(self.gates) + 1)

    self.gates = self.gates + 1

**METODO X_LIST**
- Corrisponde a `x(lista)` di Qiskit, che aggiunge una porta X per ogni qubit della lista
- È un metodo separato da `x` per evitare problemi di tipo con Nagini (un parametro che può essere sia intero sia lista)
- Chiede 1/4 del permesso sulla lista, non 1/2

**Causa**
- `_zero_reflection` ha 1/2 del permesso su `qubits`; nella prima versione `x_list` chiedeva proprio 1/2, quindi la funzione cedeva tutto il suo permesso
- Chi cede tutto il permesso, al ritorno sa della lista solo ciò che dice la postcondizione del metodo chiamato
- La postcondizione di `x_list` non dice che la lista è rimasta uguale: Nagini ne dimentica contenuto e lunghezza, e alla seconda chiamata non riesce più a dimostrare che gli indici sono validi

**Correzione**
- `x_list` (e `mcx`) chiedono 1/4; `_zero_reflection` resta a 1/2
- Così la funzione cede 1/4 e tiene l'altro 1/4: la lista non può essere modificata e Nagini continua a conoscerne il contenuto

In [ ]:
def x_list(self, qubits: List[int]) -> None:
        Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Requires(Acc(list_pred(qubits), 1/4))
        Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < self.num_qubits)))

        Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Ensures(Acc(list_pred(qubits), 1/4))
        Ensures(self.gates == Old(self.gates) + len(qubits))

        self.gates = self.gates + len(qubits)

**METODO MCX:**
- Una sola porta, qualunque sia il numero di controlli
- `len(controls) >= 1`: MCX richiede almeno un controllo; è il motivo per cui l'originale tratta a parte il caso m = 1
- Non richiede che il target sia diverso dai controlli (semplificazione, vedi 1.8)

In [ ]:
def mcx(self, controls: List[int], target: int) -> None:
        Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Requires(Acc(list_pred(controls), 1/4))
        Requires(len(controls) >= 1)
        Requires(Forall(int, lambda i: Implies(0 <= i and i < len(controls), 0 <= controls[i] and controls[i] < self.num_qubits)))
        Requires(0 <= target and target < self.num_qubits)

        Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Ensures(Acc(list_pred(controls), 1/4))
        Ensures(self.gates == Old(self.gates) + 1)

        self.gates = self.gates + 1

## 1.4 Trasformazioni

- **Parametro `mcx_mode`**
    - Originale: parametro della funzione
    - Versione verificata: rimosso, fissato a `"noancilla"`
    - Motivo: è il default di `GroverOperator`
- **Gestione delle ancille**
    - Originale: `MCXGate.get_num_ancilla_qubits(...)` e blocco `if` con `AncillaRegister`
    - Versione verificata: rimossi
    - Motivo: con `"noancilla"` le ancille sono sempre 0, quindi si prende sempre il ramo senza ancille
- **Costruzione del circuito**
    - Originale: `QuantumCircuit(QuantumRegister(n, "state"), name="S_0")`
    - Versione verificata: `QuantumCircuit(n, 0)`
    - Motivo: il modello non ha registri né nomi; il secondo argomento sono i bit classici
- **X su una lista di qubit**
    - Originale: `reflection.x(qubits)`
    - Versione verificata: `reflection.x_list(qubits)`
    - Motivo: un metodo per ogni tipo di argomento
- **Z nel caso m = 1**
    - Originale: `reflection.z(0)`
    - Versione verificata: `reflection.z(qubits[0])`
    - Motivo: correzione di un errore del codice originale (vedi 1.8)
- **Target di MCX**
    - Originale: `qubits[-1]`
    - Versione verificata: `qubits[len(qubits) - 1]`, salvato in `target`
    - Motivo: indici espliciti, non negativi
- **Controlli di MCX**
    - Originale: `qubits[:-1]`
    - Versione verificata: `qubits[:len(qubits) - 1]`, salvato in `controls`
    - Motivo: come sopra
- **Chiamata a MCX**
    - Originale: `mcx(..., qr_ancilla[:], mode=mcx_mode)`
    - Versione verificata: `mcx(controls, target)`
    - Motivo: il modello non ha ancille né modalità

**Note**
- Le riscritture di indici negativi e slicing sono preventive: la forma originale non è stata provata con Nagini
- Le variabili `target` e `controls` servono solo alla leggibilità

mcx considerata come un unica istruzione
- Sia il modello sia `size()` di Qiskit contano MCX come un'unica istruzione
- Senza ancille la scomposizione richiede molte più porte; con le ancille ne richiede meno, al prezzo di qubit in più
- Se si decidesse di contare la scomposizione, il limite dipenderebbe sia da m sia dalla modalità: è una delle decisioni aperte

## 1.5 Versione verificata

File `groverVerificato.py`.

In [ ]:
def _zero_reflection(num_state_qubits: int, qubits: List[int]) -> QuantumCircuit:
    Requires(num_state_qubits > 0)
    Requires(Acc(list_pred(qubits), 1/2))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < num_state_qubits)))

    Ensures(Acc(list_pred(qubits), 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == num_state_qubits)
    Ensures(Implies(len(qubits) == 1, Result().gates == 3))
    Ensures(Implies(len(qubits) > 1, Result().gates == 2 * len(qubits) + 3))
    Ensures(Result().num_clbits == 0)

    reflection = QuantumCircuit(num_state_qubits, 0)

    reflection.x_list(qubits)
    if len(qubits) == 1:
        reflection.z(qubits[0])
    else:
        target = qubits[len(qubits) - 1]
        controls = qubits[:len(qubits) - 1]
        reflection.h(target)
        reflection.mcx(controls, target)
        reflection.h(target)
    reflection.x_list(qubits)

    return reflection

# 2. `_state_preparation`
- Costruisce A, la preparazione di stato: una porta H su ciascun qubit di riflessione
    - corrisponde alle due trasformate di Hadamard dell'iterazione di Grover, prima e dopo $S_0$
    - in `_build` viene usata due volte: invertita (A†) prima di $S_0$ e normale (A) dopo
    - siccome $H† = H$, qui A† e A contengono le stesse porte
- Come `_zero_reflection`, non chiama altre funzioni di Grover, quindi si può verificare direttamente


## 2.1 Codice originale
File `grover_originale/grover_operator_originale.py`, righe 227-237 (Qiskit, licenza Apache 2.0). Non è una funzione libera ma una property della classe `GroverOperator`.

In [ ]:
@property
def state_preparation(self) -> QuantumCircuit:
    """The subcircuit implementing the A operator or Hadamards."""
    if self._state_preparation is not None:
        return self._state_preparation

    num_state_qubits = self.oracle.num_qubits - self.oracle.num_ancillas
    hadamards = QuantumCircuit(num_state_qubits, name="H")
    # apply Hadamards only on reflection qubits, rest will cancel out
    hadamards.h(self.reflection_qubits)
    return hadamards


- Se l'utente ha fornito una sua preparazione di stato, la restituisce così com'è
- Altrimenti calcola il numero di qubit di stato dall'oracolo e crea un circuito con H sui qubit di riflessione
- H va solo sui qubit di riflessione, perchè sugli altri qubit $S_0$ non agisce
    - quindi le $H$ di $A†$ e di $A$ si annullerebbero ($H H = I$)

## 2.3 Metodi aggiunti allo stub

**METODO H_LIST**

- Corrisponde a `h(lista)` di Qiskit: aggiunge una porta H per ogni qubit della lista
- È identico a `x_list`, cambia solo il nome
- Chiede 1/4 del permesso sulla lista per lo stesso motivo di `x_list`

In [ ]:
def h_list(self, qubits: List[int]) -> None:
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Requires(Acc(list_pred(qubits), 1/4))
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < self.num_qubits)))

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Ensures(Acc(list_pred(qubits), 1/4))
    Ensures(self.gates == Old(self.gates) + len(qubits))

    self.gates = self.gates + len(qubits)

## 2.4 Trasformazioni

- **Da property a funzione libera**
    - Originale: `@property def state_preparation(self)`
    - Versione verificata: `_state_preparation(num_state_qubits, qubits)`
    - Motivo: la property legge tutti i dati da `self` (oracolo, qubit di riflessione, preparazione dell'utente), ma il numero di porte dipende solo dal numero di qubit e dalla lista dei qubit. Passandoli come parametri si evita di modellare l'intera classe `GroverOperator` e i contratti restano piccoli
- **Preparazione fornita dall'utente**
    - Originale: `if self._state_preparation is not None: return self._state_preparation`
    - Versione verificata: rimosso
    - Motivo: una preparazione arbitraria ha un numero di porte non noto; si considera il Grover standard con le H
- **Numero di qubit di stato**
    - Originale: `self.oracle.num_qubits - self.oracle.num_ancillas`
    - Versione verificata: parametro `num_state_qubits`
    - Motivo: senza ancille coincide con il numero di qubit dell'oracolo; sarà `_build` a passarlo
- **Qubit di riflessione**
    - Originale: `self.reflection_qubits`
    - Versione verificata: parametro `qubits`
    - Motivo: come sopra; `_build` passa la stessa lista a `_state_preparation` e a `_zero_reflection`
- **Costruzione del circuito**
    - Originale: `QuantumCircuit(num_state_qubits, name="H")`
    - Versione verificata: `QuantumCircuit(num_state_qubits, 0)`
    - Motivo: il modello non ha nomi; il secondo argomento sono i bit classici
- **H su una lista di qubit**
    - Originale: `hadamards.h(self.reflection_qubits)`
    - Versione verificata: `hadamards.h_list(qubits)`
    - Motivo: un metodo per ogni tipo di argomento

**Note**

- La trasformazione è corretta perché la funzione esegue le stesse operazioni della property, a patto di riceverle con gli stessi valori che la property leggeva da `self`: `num_state_qubits` uguale al numero di qubit dell'oracolo e `qubits` uguale ai qubit di riflessione
- Regola generale, utile per il traduttore automatico: una property o un metodo che usa `self` solo in lettura diventa una funzione libera che riceve quei dati come parametri
- `_zero_reflection` non ha richiesto questo passaggio perché in Qiskit era già una funzione libera

## 2.5 Versione verificata
File `groverVerificato.py`.

In [ ]:
def _state_preparation(num_state_qubits: int, qubits: List[int]) -> QuantumCircuit:
    Requires(num_state_qubits > 0)
    Requires(Acc(list_pred(qubits), 1/2))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits),0 <= qubits[i] and qubits[i] < num_state_qubits)))

    Ensures(Acc(list_pred(qubits), 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == num_state_qubits)
    Ensures(Result().gates == len(qubits))
    Ensures(Result().num_clbits == 0)

    hadamards = QuantumCircuit(num_state_qubits, 0)
    hadamards.h_list(qubits)
    return hadamards

**Contratti**

- Le precondizioni sono le stesse di `_zero_reflection`, perché le due funzioni ricevono gli stessi dati
- Il costruttore parte da 0 porte e `h_list` ne aggiunge len(qubits): da qui `Result().gates == len(qubits)`
- `Result().num_clbits == 0` è stata aggiunta durante il livello 3: in `_build` il risultato di `_state_preparation` viene invertito con `inverse()`, che richiede `num_clbits >= 0`
- La verifica è passata al primo tentativo: le regole sui permessi ricavate al livello 1 sono bastate

# 3. `_build`

- Costruisce l'intera iterazione di Grover: Q = A S_0 A† S_f
- Mette insieme l'oracolo e le due funzioni già verificate (`_state_preparation` e `_zero_reflection`)
- Nagini non rientra nel corpo di quelle funzioni: usa solo le loro postcondizioni. Per questo andavano verificate prima
- È il primo livello in cui si combinano più circuiti, quindi servono `inverse` e `compose`


## 3.1 Codice originale

File `grover_originale/grover_operator_originale.py`, righe 244-285 (Qiskit, licenza Apache 2.0). È un metodo della classe `GroverOperator`, chiamato da `__init__`.


In [ ]:
def _build(self):
    num_state_qubits = self.oracle.num_qubits - self.oracle.num_ancillas
    circuit = QuantumCircuit(QuantumRegister(num_state_qubits, name="state"), name="Q")
    num_ancillas = numpy.max(
        [
            self.oracle.num_ancillas,
            self.zero_reflection.num_ancillas,
            self.state_preparation.num_ancillas,
        ]
    )
    if num_ancillas > 0:
        circuit.add_register(AncillaRegister(num_ancillas, name="ancilla"))

    circuit.compose(self.oracle, list(range(self.oracle.num_qubits)), inplace=True)
    if self._insert_barriers:
        circuit.barrier()
    circuit.compose(
        self.state_preparation.inverse(),
        list(range(self.state_preparation.num_qubits)),
        inplace=True,
    )
    if self._insert_barriers:
        circuit.barrier()
    circuit.compose(
        self.zero_reflection, list(range(self.zero_reflection.num_qubits)), inplace=True
    )
    if self._insert_barriers:
        circuit.barrier()
    circuit.compose(
        self.state_preparation, list(range(self.state_preparation.num_qubits)), inplace=True
    )

    # minus sign
    circuit.global_phase = numpy.pi

    self.add_register(*circuit.qregs)
    try:
        circuit_wrapped = circuit.to_gate()
    except QiskitError:
        circuit_wrapped = circuit.to_instruction()

    self.compose(circuit_wrapped, qubits=self.qubits, inplace=True)

**Cosa fa**
- Crea un circuito con i qubit di stato e, se servono, aggiunge le ancille (il massimo tra quelle di oracolo, S_0 e A)
- Compone in ordine: oracolo, A† (`state_preparation.inverse()`), S_0, A; se `insert_barriers` è vero, mette una barrier tra un blocco e l'altro
- Imposta la fase globale a π, per compensare il segno di S_0 (Qiskit implementa I - 2|0⟩⟨0| invece di 2|0⟩⟨0| - I)
- Impacchetta tutto il circuito in un unico gate con `to_gate()` e lo aggiunge a `self`



## 3.2 Limite da dimostrare


|O| + 5 porte se m = 1 e |O| + 4m + 3 porte se m ≥ 2, con |O| = porte dell'oracolo e m = len(qubits)

- L'oracolo non è costruito da Qiskit ma passato dall'utente: il limite è espresso in funzione della sua dimensione |O|
- Il conteggio si ottiene sommando i quattro blocchi:

| Blocco | m = 1 | m ≥ 2 |
| --- | --- | --- |
| oracolo S_f | \|O\| | \|O\| |
| A† (`_state_preparation` invertita) | 1 | m |
| S_0 (`_zero_reflection`) | 3 | 2m + 3 |
| A (`_state_preparation`) | 1 | m |
| **Totale** | **\|O\| + 5** | **\|O\| + 4m + 3** |

- Le due formule si riassumono in un unico limite superiore valido per ogni m ≥ 1: al massimo |O| + 4m + 3 porte (per m = 1 si ha |O| + 5 ≤ |O| + 7)
- Nel Grover standard i qubit di riflessione sono tutti i qubit di stato, quindi m = n
- Il limite riguarda il circuito "srotolato", cioè prima dell'impacchettamento con `to_gate()` (vedi le note in fondo)



## 3.3 Metodi aggiunti allo stub

**METODO INVERSE**
- Restituisce un circuito nuovo con gli stessi qubit, gli stessi bit classici e lo stesso numero di porte: l'inversa sostituisce ogni porta con la sua inversa e ribalta l'ordine
- Su `self` chiede solo 1/4 dei permessi, perché lo legge senza modificarlo
- Sul risultato dà il permesso pieno: è un circuito nuovo, che chi chiama può modificare. Nagini deduce da solo che è diverso da `self`, perché non si può avere permesso pieno su un oggetto e contemporaneamente 1/4 sullo stesso
- `num_qubits > 0` e `num_clbits >= 0` servono al costruttore usato nel corpo


In [ ]:
def inverse(self) -> 'QuantumCircuit':
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.num_clbits, 1/4) and Acc(self.gates, 1/4))
    Requires(self.num_qubits > 0 and self.num_clbits >= 0)

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.num_clbits, 1/4) and Acc(self.gates, 1/4))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == self.num_qubits)
    Ensures(Result().num_clbits == self.num_clbits)
    Ensures(Result().gates == self.gates)

    inv = QuantumCircuit(self.num_qubits, self.num_clbits)
    inv.gates = self.gates
    return inv

**METODO COMPOSE**
- Corrisponde a `compose(other, qubits, inplace=True)` di Qiskit: aggiunge a `self` le porte di `other`
- Le porte di `self` diventano le vecchie più quelle di `other`
- Permesso pieno su `self.gates`, che viene modificato; 1/4 su `other`, che viene solo letto
- `other` deve avere al massimo tanti qubit quanti `self`
- Effetto dei permessi: non si può comporre un circuito con se stesso (servirebbe più del permesso totale su `gates`); Grover non lo fa mai


In [ ]:
def compose(self, other: 'QuantumCircuit') -> None:
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Requires(Acc(other.num_qubits, 1/4) and Acc(other.gates, 1/4))
    Requires(other.num_qubits <= self.num_qubits)

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Ensures(Acc(other.num_qubits, 1/4) and Acc(other.gates, 1/4))
    Ensures(self.gates == Old(self.gates) + other.gates)

    self.gates = self.gates + other.gates

**Semplificazioni del modello**
- `compose` ignora la lista dei qubit su cui va il circuito composto e i bit classici; modella solo `inplace=True`
- `inverse` non modella le singole porte invertite, solo il loro numero


## 3.4 Trasformazioni


- **Da metodo a funzione libera**
    - Originale: `def _build(self)`, che legge oracolo, S_0 e A da `self`
    - Versione verificata: `_build(oracle, qubits)`
    - Motivo: come per `_state_preparation`, i dati letti da `self` diventano parametri
- **Numero di qubit di stato**
    - Originale: `self.oracle.num_qubits - self.oracle.num_ancillas`
    - Versione verificata: `n = oracle.num_qubits`
    - Motivo: il modello non ha ancille
- **Costruzione del circuito**
    - Originale: `QuantumCircuit(QuantumRegister(num_state_qubits, name="state"), name="Q")`
    - Versione verificata: `QuantumCircuit(n, 0)`
    - Motivo: il modello non ha registri né nomi
- **Ancille**
    - Originale: `numpy.max(...)` e `add_register(AncillaRegister(...))`
    - Versione verificata: rimossi
    - Motivo: con `mcx_mode = "noancilla"` e un oracolo senza ancille il numero di ancille è 0
- **Barrier**
    - Originale: `if self._insert_barriers: circuit.barrier()`
    - Versione verificata: rimosse
    - Motivo: il default è `insert_barriers = False`; se contarle come porte è una decisione aperta
- **Preparazione di stato e riflessione**
    - Originale: le property `self.state_preparation` (usata due volte) e `self.zero_reflection`
    - Versione verificata: `_state_preparation(n, qubits)` (chiamata due volte) e `_zero_reflection(n, qubits)`
    - Motivo: si usano le funzioni già verificate; la preparazione viene costruita due volte come nell'originale, dove la property crea un circuito nuovo a ogni accesso
- **Composizione**
    - Originale: `circuit.compose(x, list(range(x.num_qubits)), inplace=True)`
    - Versione verificata: `circuit.compose(x)`
    - Motivo: il modello non registra su quali qubit va il circuito composto
- **Fase globale**
    - Originale: `circuit.global_phase = numpy.pi`
    - Versione verificata: rimossa
    - Motivo: è una fase globale, non aggiunge porte
- **Impacchettamento**
    - Originale: `self.add_register(...)`, `circuit.to_gate()` (o `to_instruction()`) e `self.compose(circuit_wrapped, ...)`
    - Versione verificata: rimossi, la funzione restituisce direttamente `circuit`
    - Motivo: si conta il circuito srotolato, cioè il numero reale di porte; con l'impacchettamento il risultato avrebbe un'unica istruzione



## 3.5 Versione verificata

File `groverVerificato.py`.

In [ ]:
def _build(oracle: QuantumCircuit, qubits: List[int]) -> QuantumCircuit:
    Requires(Acc(oracle.num_qubits, 1/2) and Acc(oracle.gates, 1/2))
    Requires(oracle.num_qubits > 0)
    Requires(Acc(list_pred(qubits)))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < oracle.num_qubits)))

    Ensures(Acc(oracle.num_qubits, 1/2) and Acc(oracle.gates, 1/2))
    Ensures(Acc(list_pred(qubits)))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == oracle.num_qubits)
    Ensures(Implies(len(qubits) == 1, Result().gates == oracle.gates + 5))
    Ensures(Implies(len(qubits) > 1, Result().gates == oracle.gates + 4 * len(qubits) + 3))
    Ensures(Result().num_clbits == 0)
    Ensures(len(qubits) == Old(len(qubits)))

    n = oracle.num_qubits
    circuit = QuantumCircuit(n, 0)

    circuit.compose(oracle)

    a = _state_preparation(n, qubits)
    a_inv = a.inverse()
    circuit.compose(a_inv)

    s0 = _zero_reflection(n, qubits)
    circuit.compose(s0)

    a2 = _state_preparation(n, qubits)
    circuit.compose(a2)

    return circuit

**Contratti**
- **Oracolo, permesso 1/2**: viene solo letto; chi chiama si tiene il resto e sa che non è cambiato
- **`qubits`, permesso pieno**: `_build` chiama tre funzioni che chiedono 1/2 della lista ciascuna. Con solo 1/2 lo cederebbe tutto alla prima chiamata e perderebbe le informazioni sulla lista (lo stesso problema dell'errore 1). Con il permesso pieno ne cede metà e tiene l'altra metà
- **Regola generale**: chi sta più in alto nella catena di chiamate ha più permesso di chi chiama (`_build` 1, `_zero_reflection` e `_state_preparation` 1/2, metodi dello stub 1/4)
- **`Forall`**: gli indici sono validi rispetto a `oracle.num_qubits`, che è anche il numero di qubit dei circuiti creati dalle due funzioni chiamate
- **Limite**: `oracle.gates` è |O|; le due formule sono le postcondizioni con `Implies`
- **Due postcondizioni aggiunte durante il livello 4**:
    - `Result().num_clbits == 0`: serve a `power`, che richiede `num_clbits >= 0`
    - `len(qubits) == Old(len(qubits))`: chi chiama `_build` le cede tutto il permesso sulla lista, quindi senza questa postcondizione non saprebbe più che la lunghezza è rimasta la stessa. `_build` invece lo può dimostrare, perché durante le sue chiamate tiene metà del permesso

**Come Nagini arriva al limite**

| Passo | m = 1 | m ≥ 2 | Da quale postcondizione |
| --- | --- | --- | --- |
| costruttore | 0 | 0 | `__init__` |
| `compose(oracle)` | + \|O\| | + \|O\| | `compose` |
| `compose(a_inv)` | + 1 | + m | `_state_preparation`, `inverse`, `compose` |
| `compose(s0)` | + 3 | + 2m + 3 | `_zero_reflection`, `compose` |
| `compose(a2)` | + 1 | + m | `_state_preparation`, `compose` |
| **Totale** | **\|O\| + 5** | **\|O\| + 4m + 3** | |

- La verifica è passata al primo tentativo

**Note**
- **Impacchettate o srotolate**: in Qiskit `GroverOperator(oracle).size()` vale 1, perché tutto è chiuso in un unico gate; `GroverOperator(oracle).decompose().size()` apre il gate e dovrebbe dare proprio |O| + 4m + 3. Quale delle due contare è una decisione aperta
- **Riuso delle postcondizioni**: `_build` è il primo esempio di verifica modulare. Il limite di una funzione si dimostra combinando i limiti, già dimostrati, delle funzioni che chiama

# 4. `construct_circuit`

- Costruisce il circuito completo di Grover: la preparazione di stato seguita da k iterazioni
- Usa `_state_preparation` e `_build`, già verificate, più due elementi nuovi:
    - il metodo `power` nello stub, che ripete un circuito k volte
    - la funzione ausiliaria `_all_qubits`, che costruisce la lista [0, ..., n-1]
- È il livello dei cicli: `power` e `_all_qubits` contengono i primi invarianti di ciclo di Grover
- Con questo livello il limite riguarda l'intero circuito che la classe `Grover` costruisce, non più una singola iterazione


## 4.1 Codice originale

File `grover_originale/grover_originale.py`, da riga 394 (Qiskit, licenza Apache 2.0). È un metodo della classe `Grover`; la docstring è omessa.


In [ ]:
def construct_circuit(
    self, problem: AmplificationProblem, power: int | None = None, measurement: bool = False
) -> QuantumCircuit:
    if power is None:
        if len(self._iterations) > 1:
            raise ValueError("Please pass ``power`` if the iterations are not an integer.")
        power = self._iterations[0]

    qc = QuantumCircuit(problem.oracle.num_qubits, name="Grover circuit")
    qc.compose(problem.state_preparation, inplace=True)
    if power > 0:
        qc.compose(problem.grover_operator.power(power), inplace=True)

    if measurement:
        measurement_cr = ClassicalRegister(len(problem.objective_qubits))
        qc.add_register(measurement_cr)
        qc.measure(problem.objective_qubits, measurement_cr)

    return qc

**Cosa fa**
- Se il numero di iterazioni `power` non è dato, lo prende da quelle configurate nella classe
- Crea un circuito con tanti qubit quanti l'oracolo
- Compone la preparazione di stato
- Se `power > 0`, compone l'operatore di Grover ripetuto `power` volte
- Se `measurement` è vero, aggiunge un registro classico e misura i qubit obiettivo

**Dov'è il ciclo**
- Non c'è un ciclo esplicito: le k iterazioni stanno in `grover_operator.power(power)`
- In Qiskit `power` con un esponente intero chiama `repeat`, che impacchetta il circuito in un gate e lo aggiunge k volte

**I valori di default di `AmplificationProblem`** (file `grover_originale/amplification_problem_originale.py`)
- `state_preparation` (righe 95-98): se l'utente non la fornisce, è un circuito con H su tutti gli n qubit dell'oracolo
- `grover_operator` (righe 200-201): se l'utente non lo fornisce, è `GroverOperator(oracle, state_preparation)`, senza `reflection_qubits`; quindi la riflessione agisce su tutti i qubit di stato
- `objective_qubits` (righe 139-140): se non specificati, si misurano tutti gli n qubit
- Conseguenza: nel caso di default tutto agisce su tutti gli n qubit, cioè m = n e `qubits = [0, ..., n-1]`

## 4.2 Limite da dimostrare

n + k(|O| + 4n + 3) porte se n ≥ 2 e 1 + k(|O| + 5) porte se n = 1, con n = qubit dell'oracolo, |O| = porte dell'oracolo e k = numero di iterazioni

| Blocco | n = 1 | n ≥ 2 |
| --- | --- | --- |
| preparazione di stato | 1 | n |
| k iterazioni di Grover | k(\|O\| + 5) | k(\|O\| + 4n + 3) |
| **Totale** | **1 + k(\|O\| + 5)** | **n + k(\|O\| + 4n + 3)** |

- Ogni iterazione ha il numero di porte dimostrato al livello 3, con m = n
- Con k = 0 il circuito contiene solo la preparazione: n porte, e la formula dà lo stesso risultato
- Il limite dipende solo dai parametri d'ingresso: n e |O| (dall'oracolo) e k
- Le misure sono escluse: con `measurement=True` si aggiungerebbero n porte
- Come al livello 3, si conta il circuito srotolato

## 4.3 Metodi aggiunti allo stub

**METODO POWER**
- Corrisponde a `power(k)` di Qiskit: restituisce un circuito nuovo che contiene k copie di `self`
- Nel modello è srotolato: le porte del risultato sono k volte quelle di `self` (in Qiskit, con l'impacchettamento, sarebbero k istruzioni)
- Il corpo è un ciclo che compone `self` k volte in un circuito inizialmente vuoto


In [ ]:
def power(self, k: int) -> 'QuantumCircuit':
    Requires(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates, 1/2))
    Requires(self.num_qubits > 0 and self.num_clbits >= 0)
    Requires(k >= 0)

    Ensures(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates, 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == self.num_qubits)
    Ensures(Result().num_clbits == self.num_clbits)
    Ensures(Result().gates == k * self.gates)

    rep = QuantumCircuit(self.num_qubits, self.num_clbits)
    i = 0
    while i < k:
        Invariant(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates, 1/2))
        Invariant(Acc(rep.num_qubits) and Acc(rep.num_clbits) and Acc(rep.gates))
        Invariant(rep.num_qubits == self.num_qubits and rep.num_clbits == self.num_clbits)
        Invariant(0 <= i and i <= k)
        Invariant(rep.gates == i * self.gates)
        rep.compose(self)
        i = i + 1
    return rep


**Contratti e invariante**
- **Permesso 1/2 su `self`**: a ogni giro `rep.compose(self)` prende 1/4 di `self`; con 1/2, `power` tiene l'altro 1/4 e Nagini sa che `self.gates` non cambia tra un giro e l'altro (la stessa regola dell'errore 1)
- **Permessi ripetuti nell'invariante**: per Nagini il corpo del ciclo è verificato a parte e vede solo ciò che dice l'invariante, compresi i permessi
- **`rep.gates == i * self.gates`**: dopo i giri, `rep` contiene i copie di `self`. All'uscita dal ciclo vale i = k, quindi `rep.gates == k * self.gates`
- **`rep.num_qubits == self.num_qubits`**: serve alla precondizione di `compose` (`other.num_qubits <= self.num_qubits`)
- **Aritmetica non lineare**: `i * self.gates` è un prodotto tra due variabili, un tipo di formula che i dimostratori automatici a volte faticano a gestire; qui Nagini l'ha dimostrata senza aiuti


## 4.4 Funzione ausiliaria `_all_qubits`


- Costruisce la lista [0, ..., n-1], cioè i qubit di riflessione di default che Qiskit ottiene con `list(range(n))`
- È scritta con un ciclo esplicito: l'invariante dà direttamente le proprietà che servono a `_state_preparation` e `_build`, e chi la chiama ottiene il permesso pieno sulla lista
- Scelta preventiva: `list(range(n))` non è stato provato con Nagini


In [ ]:
def _all_qubits(n: int) -> List[int]:
    Requires(n > 0)
    Ensures(Acc(list_pred(Result())))
    Ensures(len(Result()) == n)
    Ensures(Forall(int, lambda j: Implies(0 <= j and j < n, Result()[j] == j)))

    qubits: List[int] = []
    i = 0
    while i < n:
        Invariant(Acc(list_pred(qubits)))
        Invariant(0 <= i and i <= n)
        Invariant(len(qubits) == i)
        Invariant(Forall(int, lambda j: Implies(0 <= j and j < i, qubits[j] == j)))
        qubits.append(i)
        i = i + 1
    return qubits


**Invariante**
- Dopo i giri la lista ha lunghezza i e contiene esattamente 0, ..., i-1
- All'uscita vale i = n, quindi la lista è [0, ..., n-1]
- Da `Result()[j] == j` Nagini ricava il `Forall` "indici validi" richiesto da `_state_preparation` e `_build`: ogni elemento è compreso tra 0 e n-1


## 4.5 Trasformazioni

- **Da metodo a funzione libera**
    - Originale: `def construct_circuit(self, problem, power, measurement)`
    - Versione verificata: `construct_circuit(oracle, k)`
    - Motivo: nel caso di default tutti i dati di `problem` si ricavano dall'oracolo; il numero di iterazioni diventa il parametro k
- **Numero di iterazioni non fornito**
    - Originale: `if power is None: ... power = self._iterations[0]`
    - Versione verificata: rimosso
    - Motivo: k è sempre passato come parametro
- **Costruzione del circuito**
    - Originale: `QuantumCircuit(problem.oracle.num_qubits, name="Grover circuit")`
    - Versione verificata: `QuantumCircuit(n, 0)` con `n = oracle.num_qubits`
    - Motivo: il modello non ha nomi; il secondo argomento sono i bit classici
- **Qubit su cui agiscono preparazione e riflessione**
    - Originale: implicito nei default (`list(range(n))` e H su tutti i qubit)
    - Versione verificata: `qubits = _all_qubits(n)`, passata esplicitamente
    - Motivo: rende esplicito il caso di default m = n
- **Preparazione di stato**
    - Originale: `problem.state_preparation`
    - Versione verificata: `_state_preparation(n, qubits)`
    - Motivo: è il default di `AmplificationProblem`, già verificato al livello 2
- **Operatore di Grover**
    - Originale: `problem.grover_operator`
    - Versione verificata: `_build(oracle, qubits)`
    - Motivo: è il default di `AmplificationProblem`, già verificato al livello 3
- **Ripetizione**
    - Originale: `problem.grover_operator.power(power)`
    - Versione verificata: `q.power(k)`, srotolato
    - Motivo: si conta il numero reale di porte, coerentemente con il livello 3
- **Misure**
    - Originale: blocco `if measurement:` con `ClassicalRegister` e `measure`
    - Versione verificata: rimosso
    - Motivo: il default è `measurement=False`; il modello non ha registri classici né misure su liste
- **Composizione**
    - Originale: `qc.compose(x, inplace=True)`
    - Versione verificata: `qc.compose(x)`
    - Motivo: il modello prevede solo `inplace=True`

## 4.6 Versione verificata

File `groverVerificato.py`.

In [ ]:
def construct_circuit(oracle: QuantumCircuit, k: int) -> QuantumCircuit:
    Requires(Acc(oracle.num_qubits) and Acc(oracle.gates))
    Requires(oracle.num_qubits > 0)
    Requires(k >= 0)

    Ensures(Acc(oracle.num_qubits) and Acc(oracle.gates))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == oracle.num_qubits)
    Ensures(Implies(oracle.num_qubits == 1,
                    Result().gates == 1 + k * (oracle.gates + 5)))
    Ensures(Implies(oracle.num_qubits > 1,
                    Result().gates == oracle.num_qubits + k * (oracle.gates + 4 * oracle.num_qubits + 3)))

    n = oracle.num_qubits
    qubits = _all_qubits(n)

    qc = QuantumCircuit(n, 0)
    prep = _state_preparation(n, qubits)
    qc.compose(prep)

    if k > 0:
        q = _build(oracle, qubits)
        qk = q.power(k)
        qc.compose(qk)

    return qc


**Contratti**
- **Oracolo, permesso pieno**: `construct_circuit` è in cima alla catena di chiamate. `_build` prende 1/2 dell'oracolo e `construct_circuit` tiene l'altra metà, così `oracle.gates` resta noto fino alla postcondizione
- **`k >= 0`**: serve a `power`
- **Limite**: espresso solo con `oracle.num_qubits` (n), `oracle.gates` (|O|) e k, cioè i parametri d'ingresso
- **La lista `qubits`**: è creata dentro la funzione, con permesso pieno
    - `_state_preparation` ne prende 1/2: la funzione tiene l'altra metà e continua a conoscerne lunghezza e contenuto
    - `_build` la prende tutta: per questo `_build` garantisce `len(qubits) == Old(len(qubits))`, che permette a Nagini di sostituire `len(qubits)` con n nel limite di `_build`
- **`power` su `q`**: richiede `q.num_clbits >= 0`, garantito dalla postcondizione `num_clbits == 0` aggiunta a `_build`

**Come Nagini arriva al limite** (caso n ≥ 2)

| Passo | Porte di `qc` | Da quale postcondizione |
| --- | --- | --- |
| costruttore | 0 | `__init__` |
| `compose(prep)` | n | `_state_preparation`, `compose` |
| `q = _build(oracle, qubits)` | (q ha \|O\| + 4n + 3 porte) | `_build` |
| `qk = q.power(k)` | (qk ha k(\|O\| + 4n + 3) porte) | `power` |
| `compose(qk)` | n + k(\|O\| + 4n + 3) | `compose` |

- Se k = 0 il ramo `if` non viene eseguito e restano n porte, che coincide con la formula
- Il caso n = 1 è identico, con |O| + 5 porte per iterazione
- La verifica è passata dopo aver aggiunto a `_build` le due postcondizioni `num_clbits == 0` e `len(qubits) == Old(len(qubits))`

**Note**
- **Impacchettate o srotolate**: in Qiskit il circuito costruito da `construct_circuit` ha n + k istruzioni (n porte H più k copie impacchettate dell'operatore); il modello conta invece n + k(|O| + 4n + 3) porte. Quale delle due contare è una decisione aperta
- **Cosa copre il risultato**: il circuito completo di Grover, per ogni k ≥ 0, nel caso di default (preparazione con H su tutti i qubit, riflessione su tutti i qubit, `mcx_mode = "noancilla"`, oracolo dato come circuito, nessuna misura)
- **Cosa non copre**:
    - la scelta del numero ottimale di iterazioni (circa \frac{\pi}{4}\sqrt{N/M}), che spetta a chi usa `Grover`
    - la parte di esecuzione (`amplify`), che non costruisce circuiti